# Course 5 · Python Project for Data Science
The course project: **(A)** get stock prices from an API (`yfinance`), **(B)** scrape data from a web page (`BeautifulSoup`), **(C)** plot both in a dashboard.
*Needs internet. Every line has a `#` comment.*

## Step 1 · Imports

In [ ]:
import pandas as pd, requests                 # tables; downloading web pages
from io import StringIO                        # wraps text so pandas can read it like a file
from bs4 import BeautifulSoup                  # parses HTML so we can search it
import yfinance as yf                          # Yahoo Finance API wrapper
import plotly.graph_objects as go              # low-level Plotly building blocks (traces)
from plotly.subplots import make_subplots      # a figure with several panels

## Step 2 · API extraction with yfinance

In [ ]:
tesla = yf.Ticker("TSLA")                       # an object for one stock symbol
price = tesla.history(period="5y")             # download 5 years of daily prices (a DataFrame)
price = price.reset_index()                    # move the Date index into a normal column
price["Date"] = price["Date"].dt.tz_localize(None)   # drop the time zone so dates plot cleanly
price[["Date", "Open", "Close", "Volume"]].tail()    # show the last 5 rows of four columns

## Step 3 · Download a web page

In [ ]:
url = "https://en.wikipedia.org/wiki/Regions_of_Ethiopia"
# A User-Agent header identifies your program; some sites refuse requests without one.
headers = {"User-Agent": "Mozilla/5.0 (learning project)"}
html = requests.get(url, headers=headers, timeout=20).text   # .text = the page's HTML as a string
soup = BeautifulSoup(html, "html.parser")    # turn the HTML text into a searchable tree
print(soup.title.text)                       # the text inside the page's <title> tag

## Step 4 · Scrape a table manually (the course method)
HTML tables are built from `<table>` → rows `<tr>` → cells `<th>` (header) / `<td>` (data).

In [ ]:
table = soup.find("table", class_="wikitable")    # first <table> with class "wikitable"
rows = []                                         # empty list to collect each row
for tr in table.find_all("tr"):                   # loop over every row in the table
    # For each header or data cell in the row, take its visible text without extra spaces.
    cells = [c.get_text(strip=True) for c in tr.find_all(["th", "td"])]
    if cells:                                     # skip empty rows
        rows.append(cells)
header, body = rows[0], rows[1:]                  # first row = column names; the rest = data
scraped = pd.DataFrame(body)                      # build a table from the list of rows
print("Header:", header)
scraped.head()

## Step 5 · The shortcut: `pd.read_html`
Always check how many tables were found before choosing one.

In [ ]:
tables = pd.read_html(StringIO(html))        # parse EVERY <table> on the page into a list of DataFrames
print(len(tables), "tables found")           # how many? positions can change when the page is edited
tables[0].head()                             # look at the first one (index 0)

## Step 6 · Clean scraped text into numbers
Scraped values come as text like `"$25,500"`. Remove symbols and convert.

In [ ]:
# Sample revenue rows for practice (in the real lab you scraped these from a page).
revenue = pd.DataFrame({
    "Date": ["2024-03-31", "2024-06-30", "2024-09-30", "2024-12-31", "2025-03-31", "2025-06-30"],
    "Revenue": ["$21,301", "$25,500", "$25,182", "$25,707", "$19,335", "$22,496"],
})
# .str.replace with a regular expression: [$,] means "any $ or comma" -> replace with nothing.
# .astype(float) then converts the cleaned text into decimal numbers.
revenue["Revenue"] = revenue["Revenue"].str.replace(r"[$,]", "", regex=True).astype(float)
revenue["Date"] = pd.to_datetime(revenue["Date"])   # text -> real dates (needed for a time axis)
revenue.dtypes                                     # check: Date is datetime64, Revenue is float64

## Step 7 · Two-panel dashboard

In [ ]:
# 2 rows x 1 column of charts sharing the same x-axis (dates).
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12,
                    subplot_titles=("Share price (USD)", "Quarterly revenue (USD millions)"))
# Add a line (Scatter trace) of closing price to the top panel...
fig.add_trace(go.Scatter(x=price.Date, y=price.Close, name="Close"), row=1, col=1)
# ...and revenue to the bottom panel.
fig.add_trace(go.Scatter(x=revenue.Date, y=revenue.Revenue, name="Revenue"), row=2, col=1)
fig.update_layout(height=650, title="Tesla: price vs revenue", showlegend=False)  # size and title
fig.show()                                    # draw it (interactive: hover, zoom)

## ✍️ Your turn
1. Repeat Steps 2 and 7 for GameStop (`GME`), like the course.
2. Scrape a different table from any Wikipedia page into a clean DataFrame. Check the site's terms first.

In [ ]:
# your answer here